[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/27_vit_patch.ipynb)

# 🟠 Medium: ViT Patch Embedding

Implement the **patch embedding** layer from Vision Transformer (ViT).

### Signature
```python
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        # Returns: (B, num_patches, embed_dim)
```

### Algorithm
1. Reshape image into non-overlapping patches: `(B, C, H, W)` → `(B, N, C*P*P)`
2. Project each patch: `nn.Linear(C*P*P, embed_dim)`
3. `num_patches = (img_size // patch_size) ** 2`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [3]:

import torch
import torch.nn as nn

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        # pass  # self.num_patches, self.proj
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        self.num_patches = (img_size//patch_size)**2
        self.project_layer = nn.Linear(in_channels * patch_size * patch_size, embed_dim)



    def forward(self, x):
        # pass  # reshape to patches, project
        B, C, H, W = x.shape
        P = self.patch_size

        x_unfold = x.unfold(2, P, P).unfold(3, P, P)
        # B, C, H//P, W//P, P, P -> B, H//P, W//P, C, P, P
        x_perm = x_unfold.permute(0, 2, 3, 1, 4, 5)
        x_comp = x_perm.reshape(B, self.num_patches, C*P*P)


        # using reshape: # split H and W into (num_patches × patch_size)
        # x = x.reshape(B, C, H // P, P, W // P, P)   # (B, C, H/P, P, W/P, P)
        # x = x.permute(0, 2, 4, 1, 3, 5)             # (B, H/P, W/P, C, P, P)
        # x = x.reshape(B, self.num_patches, C * P * P)  # (B, N, C*P*P)
        output = self.project_layer(x_comp)
        return output

In [9]:
# 🧪 Debug
pe = PatchEmbedding(32, 8, 3, 64)
x = torch.randn(2, 3, 32, 32)
print('Output:', pe(x).shape)
print('Patches:', pe.num_patches)

Output: torch.Size([2, 16, 64])
Patches: 16


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('vit_patch')


🧪 Testing: ViT Patch Embedding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.6ms)
  ✅ [2/4] num_patches attribute (3.1ms)
  ✅ [3/4] Different image sizes (0.4ms)
  ✅ [4/4] Gradient flow (2.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.0ms total)
  Progress saved. Run status() to see your dashboard.

